# 🌌 Gaussian Processes: Non-Parametric Bayesian Regression

A Gaussian Process (GP) is a collection of random variables, any finite number of which have a joint Gaussian distribution. Instead of inferring a distribution over *parameters* (like in Bayesian Linear Regression), a GP infers a distribution over *functions* directly.

### Mathematical Foundation
A GP is completely specified by its mean function and covariance (kernel) function:
$$f(x) \sim \mathcal{GP}(m(x), k(x, x'))$$

Given noisy observations $\mathbf{y} = f(X) + \epsilon$ where $\epsilon \sim \mathcal{N}(0, \sigma_y^2)$, the joint distribution over train ($X, \mathbf{y}$) and test ($X_*, \mathbf{f}_*$) points gives the posterior predictive equations:

$$\mu_* = K_*^T (K + \sigma_y^2 I)^{-1} \mathbf{y}$$
$$\Sigma_* = K_{**} - K_*^T (K + \sigma_y^2 I)^{-1} K_*$$

Where $K = k(X, X), K_* = k(X, X_*)$, and $K_{**} = k(X_*, X_*)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ml_from_scratch.advanced import GaussianProcessRegressor

np.random.seed(42)
plt.style.use('ggplot')

## 1. Defining Toy Data
Let's sample from a non-linear sine function with noise.

In [ ]:
def true_function(x):
    return np.sin(3 * x) + 0.1 * x

X_train = np.random.uniform(-3, 3, 10).reshape(-1, 1)
y_train = true_function(X_train).ravel() + np.random.normal(0, 0.2, 10)

X_test = np.linspace(-4, 4, 100).reshape(-1, 1)
y_true = true_function(X_test).ravel()

## 2. Fitting the Gaussian Process
We use the Squared Exponential (RBF) kernel, which enforces function smoothness.

In [ ]:
gp = GaussianProcessRegressor(length_scale=1.0, sigma_f=1.0, sigma_y=0.2)
gp.fit(X_train, y_train)

mu_s, std_s = gp.predict(X_test, return_std=True)

## 3. Visualizing Confidence Intervals
Notice how the uncertainty (shaded region) grows where there is no data.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(X_test, y_true, 'b--', label='True Function')
plt.plot(X_train, y_train, 'ko', label='Observations')
plt.plot(X_test, mu_s, 'r-', label='GP Mean')
plt.fill_between(X_test.ravel(), mu_s - 1.96 * std_s, mu_s + 1.96 * std_s, 
                 alpha=0.2, color='red', label='95% Confidence')
plt.legend()
plt.title('Gaussian Process Regression')
plt.show()